In [ ]:
import re

def clean_text(text: str, remove_numbers: bool = True) -> str:
    # 1. Loại bỏ HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # 2. Loại bỏ URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # 3. Loại bỏ emoji mặt cười (:) ), :D, v.v.)
    text = re.sub(r"[:;=xX][-~]?[)DPp(]*", " ", text)  

    # 4. Loại bỏ số (nếu bật remove_numbers)
    if remove_numbers:
        text = re.sub(r"\d+", " ", text)
    # 4.1. Loại bỏ ký tự đặc biệt
    text = re.sub(r'[!@#$%^&\.]*',"",text)

    # 5. Chuyển về chữ thường
    text = text.lower()

    # 6. Xóa khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()
    return text


# --- Test ---
raw_text = "Sản phẩm này <br> thật tuyệt vời! Mua ngay tại <a href='https://example.com'> đây</a>. Gía chỉ 99$! :)) "
print(clean_text(raw_text,False))


sản phẩm này thật tuyệt vời mua ngay tại đây gía chỉ 99


In [7]:
import re

dict_vn = ["hôm nay", "thứ tư", "tư", "là", "tôi", "chúng tôi", "có", "tiết", "tiết học", "học", "về",
 "phân", "tích", "phân tích", "dữ", "liệu", "dữ liệu", "văn", "bản", "văn bản", "khá", "dở", "tuy nhiên", "vẫn", "còn",
   "nhiều", "thứ", "hay ho", "hay", "ho", "và", "chưa", "hiểu", "hết", "chưa hiểu"]
dict_vn = list(set(dict_vn))

# Stopwords ví dụ (bạn có thể mở rộng thêm)
stopwords = ["là", "có", "và", "về", "những", "nhiều", "vẫn" ,"còn"]
# Văn bản test
text = "Hôm nay là thứ tư. Chúng tôi có tiết học về phân tích dữ liệu văn bản. Tiết học khá dở tuy nhiên vẫn còn nhiều thứ hay ho và chưa hiểu hết"

# ----------------------
# 1. Longest Matching (tách từ dài nhất trước)
# để tìm từ có >= 2 chữ, thì ta đi lùi => Giống như loại bỏ phần tử trong list nếu ko muốn lệch index
def longest_matching(sentence, dictionary):
    tokens = []; words = sentence.lower().split(); i = 0
    while i < len(words):
        match = None
        for j in range(len(words), i, -1):
            phrase = " ".join(words[i:j])
            if phrase in dictionary:
                match = phrase; tokens.append(match); i = j ; break
        if not match:  # nếu không khớp thì lấy 1 từ
            tokens.append(words[i]); i += 1
    return tokens

# ----------------------
# 2. Maximum Matching (tách từ từ trái sang, greedy)
def maximum_matching(sentence, dictionary):
    tokens = []; words = sentence.lower().split() ; i = 0
    while i < len(words):
        match = None
        for j in range(len(words), i, -1):
            phrase = " ".join(words[i:j])
            if phrase in dictionary:
                match = phrase ; break
        if match:
            tokens.append(match) ; i += len(match.split())
        else:
            tokens.append(words[i]) ; i += 1
    return tokens


In [ ]:
# ----------------------
# 3. VNCoreNLP (nếu bạn đã cài VNCoreNLP)
# pip install vncorenlp
# import vncorenlp
# vncorenlp.download_model(save_dir='./vncorenlp')
# rdrsegmenter = vncorenlp.VnCoreNLP("./vncorenlp/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx2g')
# tokens_vncore = rdrsegmenter.tokenize(text)

# ----------------------
# Loại bỏ stopwords
def remove_stopwords(tokens, stopwords):
    return [t for t in tokens if t not in stopwords]

# ----------------------
# Test
tokens_longest = longest_matching(text, dict_vn)
tokens_maximum = maximum_matching(text, dict_vn)

print("Longest Matching:", remove_stopwords(tokens_longest, stopwords))
print("Maximum Matching:", remove_stopwords(tokens_maximum, stopwords))
# print("VNCoreNLP:", remove_stopwords([w for sent in tokens_vncore for w in sent], stopwords))

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from pyvi import ViTokenizer
import re

# ==================== STEMMING với NLTK ====================
print("=" * 60)
print("STEMMING - Thư viện: NLTK (PorterStemmer)")
print("=" * 60)

# Câu gốc
sentence = "Hnay là thứ tu. Chúng tôi có tiết học về phân tích và xu li dữ liệu văn bản. Tiếtt học khá thú viii, tuy nhiên vẫn còn nhiều thứ chưa hiểu hết....."

print("\nCâu gốc:")
print(sentence)

# Tokenize với pyvi (vì NLTK không tốt cho tiếng Việt)
tokens_pyvi = ViTokenizer.tokenize(sentence)
print("\nSau khi tokenize (pyvi):")
print(tokens_pyvi)

# Stemming với NLTK PorterStemmer
stemmer = PorterStemmer()

# Chuẩn bị từ cho stemming
words_for_stem = tokens_pyvi.split()
words_for_stem = [word.lower() for word in words_for_stem]
words_for_stem = [re.sub(r'[^a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', '', word) for word in words_for_stem]
words_for_stem = [word for word in words_for_stem if word]

# Áp dụng stemming
stemmed_words = [stemmer.stem(word) for word in words_for_stem]

print("\nSau stemming (NLTK PorterStemmer):")
print(" ".join(stemmed_words))


# ==================== LEMMATIZATION với NLTK ====================
print("LEMMATIZATION - Thư viện: NLTK (WordNetLemmatizer)")

tokens_pyvi_lem = ViTokenizer.tokenize(sentence)
print("Sau khi tokenize (pyvi):")
print(tokens_pyvi_lem)

# Lemmatization với NLTK WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# Chuẩn bị từ cho lemmatization
words_for_lem = tokens_pyvi_lem.split()
words_for_lem = [word.lower() for word in words_for_lem]
words_for_lem = [re.sub(r'[^a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', '', word) for word in words_for_lem]
words_for_lem = [word for word in words_for_lem if word]

# Áp dụng lemmatization
lemmatized_words = [lemmatizer.lemmatize(word) for word in words_for_lem]

print("Sau lemmatization (NLTK WordNetLemmatizer):")
print(" ".join(lemmatized_words))


# ==================== SO SÁNH ====================
print("KẾT QUẢ SO SÁNH")

print("\n📌 STEMMING (NLTK - PorterStemmer):")
print(f"   Câu gốc: {sentence}")
print(f"   Kết quả: {' '.join(stemmed_words)}")

print("\n📌 LEMMATIZATION (NLTK - WordNetLemmatizer):")
print(f"   Câu gốc: {sentence}")
print(f"   Kết quả: {' '.join(lemmatized_words)}")

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger'); nltk.download('maxent_ne_chunker'); nltk.download('words')
import stanza
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyvi import ViTokenizer
from nltk import pos_tag, ne_chunk
from nltk.tree import Tree
import re
# Khởi tạo Stanza pipeline cho tiếng Việt
print("Đang tải Stanza pipeline...")
nlp = stanza.Pipeline(lang='vi', processors='tokenize,mwt,pos,lemma,depparse,constituency')
# Câu cần phân tích
sentence = "Các sinh viên Khoa học máy tính ở PTIT đang học Phân tích dữ liệu tại cơ sở Ngọc Trục"

# Xử lý với Stanza
doc = nlp(sentence)
# ==================== 1. POS TAGGING ====================
print("1. GÁN NHÃN TỪ LOẠI (POS TAGGING)")
pos_tags = []
for token in doc.sentences[0].tokens:
    for word in token.words:
        pos_tags.append((word.text, word.pos))
        print(f"  {word.text:<15} → {word.pos}")

# ==================== 2. CHUNKING (NP, VP) ====================
print("2. PHÂN TÍCH CỤM TỪ (CHUNKING) - NP và VP")

def find_chunks(tokens, pos_tags_list):
    """Tìm các cụm danh từ (NP) và cụm động từ (VP)"""
    pos_dict = {text: pos for text, pos in pos_tags_list}
    np_chunks = []; vp_chunks = []; current_chunk = []; chunk_type = None
    for text, pos in pos_tags_list:
        if pos.startswith('N'):  # Danh từ
            if chunk_type == 'NP':
                current_chunk.append(text)
            else:
                if current_chunk:
                    if chunk_type == 'NP':
                        np_chunks.append(' '.join(current_chunk))
                    elif chunk_type == 'VP':
                        vp_chunks.append(' '.join(current_chunk))
                current_chunk = [text]
                chunk_type = 'NP'
        elif pos.startswith('V'):  # Động từ
            if chunk_type == 'VP':
                current_chunk.append(text)
            else:
                if current_chunk:
                    if chunk_type == 'NP':
                        np_chunks.append(' '.join(current_chunk))
                    elif chunk_type == 'VP':
                        vp_chunks.append(' '.join(current_chunk))
                current_chunk = [text]
                chunk_type = 'VP'
        elif pos in ['A', 'ADV'] and current_chunk:  # Tính từ, trạng từ
            current_chunk.append(text)
    
    if current_chunk:
        if chunk_type == 'NP':
            np_chunks.append(' '.join(current_chunk))
        elif chunk_type == 'VP':
            vp_chunks.append(' '.join(current_chunk))
    
    return np_chunks, vp_chunks

np_chunks, vp_chunks = find_chunks(doc.sentences[0].tokens, pos_tags)

print("\n🔹 Cụm Danh Từ (NP - Noun Phrase):")
for i, chunk in enumerate(set(np_chunks), 1):
    print(f"  {i}. {chunk}")

print("\n🔹 Cụm Động Từ (VP - Verb Phrase):")
for i, chunk in enumerate(set(vp_chunks), 1):
    print(f"  {i}. {chunk}")

# ==================== 3. DEPENDENCY PARSING ====================
print("3. PHÂN TÍCH CỤ PHÁP PHỤ THUỘC (DEPENDENCY PARSING)")
print("\n🔹 Quan hệ phụ thuộc giữa các từ:")
dep_relations = []
for token in doc.sentences[0].tokens:
    for word in token.words:
        if word.head != 0:
            head_word = doc.sentences[0].words[word.head - 1]
            relation = f"{word.text} --[{word.deprel}]--> {head_word.text}"
            dep_relations.append(relation)
            print(f"  {relation}")

# ==================== 4. CONSTITUENCY PARSING ====================
print("4. PHÂN TÍCH CỤ PHÁP THÀNH PHẦN (CONSTITUENCY PARSING)")
# Lấy cây thành phần từ Stanza
const_tree = doc.sentences[0].constituency # type: ignore
print("\n🔹 Cây thành phần (Constituency Tree):")
print(const_tree)

def print_tree(tree, level=0):
    """In cây thành phần một cách dễ đọc"""
    indent = "  " * level
    if isinstance(tree, Tree):
        print(f"{indent}[{tree.label()}]")
        for child in tree:
            print_tree(child, level + 1)
    else:
        print(f"{indent}{tree}")

print("\n🔹 Biểu diễn cây thành phần (Tree format):")
print_tree(const_tree)

# ==================== 5. TRỰC QUAN HÓA ====================
print("5. TRỰC QUAN HÓA")
# Vẽ cây phụ thuộc
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
# Cây phụ thuộc
ax1.set_title("Dependency Parsing Tree", fontsize=14, fontweight='bold')
ax1.text(0.5, 0.95, sentence, ha='center', va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7),transform=ax1.transAxes)
y_pos = 0.85
for relation in dep_relations[:10]:  # Hiển thị 10 quan hệ đầu tiên
    ax1.text(0.1, y_pos, relation, fontsize=9, family='monospace', transform=ax1.transAxes)
    y_pos -= 0.08
ax1.axis('off')

# Biểu đồ từ loại
ax2.set_title("POS Tags Distribution", fontsize=14, fontweight='bold')
pos_dict = {}
for word, pos in pos_tags:
    pos_dict[pos] = pos_dict.get(pos, 0) + 1

colors = plt.cm.Set3(range(len(pos_dict))) # type: ignore
wedges, texts, autotexts = ax2.pie(pos_dict.values(), labels=pos_dict.keys(), autopct='%1.1f%%', colors=colors, startangle=90)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.savefig('nlp_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Biểu đồ đã được lưu vào 'nlp_analysis.png'")
plt.show()

# ==================== TÓMLẠI ====================
print("TỔNG KẾT")
print(f"""
✓ Tổng số từ: {len(pos_tags)}
✓ Số cụm danh từ: {len(set(np_chunks))}
✓ Số cụm động từ: {len(set(vp_chunks))}
✓ Quan hệ phụ thuộc: {len(dep_relations)}
✓ Từ loại chính: {max(pos_dict.items(), key=lambda x: x[1])[0]}
""")